In [ ]:
import numpy as np
# import pandas as pd
import matplotlib.pyplot as plt
# import seaborn as sns
# from ripser import ripser
# from persim import PersistenceImager, plot_diagrams
# from persim.persistent_entropy import persistent_entropy
# from gtda.diagrams import BettiCurve
from tda_methods import GeometryConverter, PersistenceAnalysis


In [ ]:
rng = np.random.default_rng()
start_point, end_point, n_points = 0, 2*np.pi, 1000
u   = rng.uniform(start_point, end_point, n_points)
v   = rng.uniform(start_point, end_point, n_points)

# u, v = np.meshgrid(u, v)

R_major   = 1
r_tube    = R_major/4
converter = GeometryConverter(R_major, r_tube)
x, y, z   = converter.convert_angles_to_torus_xyz(u, v)
# # x, y, z   = converter.convert_angles_to_sphere_xyz(u, v)
noise = rng.normal(0, 0.04, n_points)
x += noise  # Add some noise to v
y += noise  # Add some noise to u
z += noise  # Add some noise to both
geo_coordinates = np.column_stack((x, y, z))  # shape (N, 3)

# x1,y1,z1,x2,y2,z2 = converter.convert_angles_to_2torus_xyz(u, v, d_shift=R_major)
# geo_coordinates   = np.vstack([np.column_stack((x1,y1,z1)), np.column_stack((x2,y2,z2))])


In [ ]:

def plot_3d_points(*clouds, colors=None, figsize=(8, 12), size=3, alpha=0.7):
    """Plot one or multiple 3D point clouds with equal axis scaling.
    clouds: tuples of (x, y, z)
    colors: list of colors (optional)"""
    fig = plt.figure(figsize=figsize)
    ax  = fig.add_subplot(projection='3d')

    if colors is None:
        colors = ['red'] * len(clouds)

    all_x = np.concatenate([c[0] for c in clouds])
    all_y = np.concatenate([c[1] for c in clouds])
    all_z = np.concatenate([c[2] for c in clouds])

    max_range = np.array([
        all_x.max() - all_x.min(),
        all_y.max() - all_y.min(),
        all_z.max() - all_z.min()]).max() / 2.0

    mid_x = (all_x.max() + all_x.min()) * 0.5
    mid_y = (all_y.max() + all_y.min()) * 0.5
    mid_z = (all_z.max() + all_z.min()) * 0.5

    for (x, y, z), c in zip(clouds, colors):
        ax.scatter(x, y, z, color=c, alpha=alpha, s=size)

    ax.set_xlim(mid_x - max_range, mid_x + max_range)
    ax.set_ylim(mid_y - max_range, mid_y + max_range)
    ax.set_zlim(mid_z - max_range, mid_z + max_range)

    plt.margins(0)
    plt.show()

plot_3d_points((geo_coordinates[:, 0], geo_coordinates[:, 1], geo_coordinates[:, 2]))

In [ ]:
# persistence stuff

persistence    = PersistenceAnalysis(max_dim=1)
diagrams_list  = persistence.plot_persistence_diagrams(geo_coordinates, to_plot=True)
diagrams_clean = persistence.remove_inf(diagrams_list)
entropy_array  = persistence.compute_entropy(diagrams_clean)
persistence_images_list = persistence.compute_persistence_image(diagrams_clean)
betti_curves_array      = persistence.compute_betti_curves(diagrams_clean)
diagrams_clean          = persistence.remove_inf(diagrams_list)

print("Persistence array:", entropy_array)
print("Betti curves shape: (batch, homology_dim, filtration_steps) =", betti_curves_array.shape)

persistence.plot_entropy(entropy_array)
persistence.plot_persistence_image(persistence_images_list[1])  # H1
persistence.plot_betti_curves(betti_curves_array)
